In [1]:
import numpy as np
import struct
import os
from tqdm import tqdm

In [2]:
def read_bin_header(filepath):
    """Read header from .fbin, .u8bin, or .i8bin file."""
    
    ext_to_dtype = {
        '.fbin':  np.float32,
        '.u8bin': np.uint8,
        '.i8bin': np.int8,
    }
    
    ext = os.path.splitext(filepath)[1].lower()
    dtype = ext_to_dtype.get(ext)
    if dtype is None:
        raise ValueError(f"Unknown extension '{ext}'. Expected .fbin, .u8bin, or .i8bin")
    
    with open(filepath, 'rb') as f:
        num_points, num_dimensions = struct.unpack('<II', f.read(8))
    
    expected_data_bytes = num_points * num_dimensions * np.dtype(dtype).itemsize
    actual_data_bytes = os.path.getsize(filepath) - 8
    
    return {
        'num_points':     num_points,
        'num_dimensions': num_dimensions,
        'dtype':          dtype,
        'expected_bytes': expected_data_bytes,
        'actual_bytes':   actual_data_bytes,
        'size_match':     expected_data_bytes == actual_data_bytes,
    }

In [12]:
file = "/scratch/pa2439/ANN-Search/datasets/yandex_deep/base.1B.fbin"
read_bin_header(file)

{'num_points': 1000000000,
 'num_dimensions': 96,
 'dtype': numpy.float32,
 'expected_bytes': 384000000000,
 'actual_bytes': 384000000000,
 'size_match': True}

In [13]:
def process_bin_file(filepath, output_dir=".", batch_size=1_000_000):
    ext_to_dtype = {
        '.fbin':  np.float32,
        '.u8bin': np.uint8,
        '.i8bin': np.int8,
    }

    ext = os.path.splitext(filepath)[1].lower()
    dtype = ext_to_dtype.get(ext)
    if dtype is None:
        raise ValueError(f"Unknown extension '{ext}'. Expected .fbin, .u8bin, or .i8bin")

    with open(filepath, 'rb') as f:
        num_points, num_dimensions = struct.unpack('<II', f.read(8))

        ids_path      = os.path.join(output_dir, 'ids.npy')
        vectors_path  = os.path.join(output_dir, 'vectors.npy')
        sq_norms_path = os.path.join(output_dir, 'sq_norms.npy')

        ids_out      = np.lib.format.open_memmap(ids_path,      mode='w+', dtype=np.int32,   shape=(num_points,))
        vectors_out  = np.lib.format.open_memmap(vectors_path,  mode='w+', dtype=dtype,       shape=(num_points, num_dimensions))
        sq_norms_out = np.lib.format.open_memmap(sq_norms_path, mode='w+', dtype=np.float32,  shape=(num_points,))

        for start in tqdm(range(0, num_points, batch_size)):
            end = min(start + batch_size, num_points)
            batch = np.fromfile(f, dtype=dtype, count=(end - start) * num_dimensions)
            batch = batch.reshape(end - start, num_dimensions)

            ids_out[start:end]      = np.arange(start, end, dtype=np.int32)
            vectors_out[start:end]  = batch
            sq_norms_out[start:end] = np.sum(batch.astype(np.float32) ** 2, axis=1)

            # print(f"  processed {end}/{num_points} ({100*end/num_points:.1f}%)")

    print(f"Done. {num_points} vectors of dimension {num_dimensions} ({dtype.__name__})")

In [14]:
process_bin_file(
    filepath=file,
    output_dir="/scratch/pa2439/ANN-Search/datasets/yandex_deep",
    batch_size=1_000_000
)

100%|██████████| 1000/1000 [33:11<00:00,  1.99s/it]


Done. 1000000000 vectors of dimension 96 (float32)
